# Zadanie 1 -- Perceptron AND i OR
Zaimplementuj perceptron od zera i naucz go bramek logicznych AND i OR.

Wymagania:
- Klasa Perceptron z metodami fit() i predict()
- Trening na AND: (0,0)=0, (0,1)=0, (1,0)=0, (1,1)=1
- Trening na OR: (0,0)=0, (0,1)=1, (1,0)=1, (1,1)=1
- Wyświetl wagi i bias dla każdej bramki
- Zweryfikuj predykcje

Oczekiwany wynik:
- Poprawne predykcje dla AND i OR

In [ ]:
# Perceptron od zera -- bramki logiczne AND i OR
import numpy as np  # biblioteka do obliczeń na macierzach i wektorach


class Perceptron:
    """Najprostszy model neuronu -- klasyfikator liniowy (przesuwa linię decyzyjną)."""

    def __init__(self, n_features, learning_rate=0.1):
        # n_features = liczba cech wejściowych (tu: 2 wejścia bramki logicznej)
        # learning_rate = jak duży krok robimy przy poprawianiu wag w każdej iteracji
        self.weights = np.random.randn(n_features) * 0.01  # wagi W -- startują blisko zera
        self.bias = 0.0  # bias b -- przesunięcie progu decyzyjnego
        self.lr = learning_rate  # zapisujemy learning rate do późniejszego użycia w fit()

    def activation(self, z):
        # z = w1*x1 + w2*x2 + ... + b  (suma ważona + bias)
        # funkcja progowa: jeśli z >= 0 --> 1, inaczej --> 0
        return np.where(z >= 0, 1, 0)

    def predict(self, X):
        """Predykcja dla wielu próbek naraz: y = step(X @ W + b)."""
        z = X @ self.weights + self.bias  # oblicz sumę ważoną dla każdego wiersza X
        return self.activation(z)  # zamień wynik liczbowy na klasę 0 lub 1

    def fit(self, X, y, n_epochs=100):
        """Trening -- reguła uczenia Rosenblatta (poprawiaj wagę tylko przy błędzie)."""
        self.errors_per_epoch = []  # lista: ile błędów było w każdej epoce

        for epoch in range(n_epochs):  # epoka = jeden pełny przejście przez wszystkie próbki
            errors = 0  # licznik błędów w tej epoce

            for xi, yi in zip(X, y):  # xi = jedna próbka wejściowa, yi = poprawna odpowiedź
                y_pred = self.activation(xi @ self.weights + self.bias)  # predykcja dla xi
                error = yi - y_pred  # błąd: 0 (trafione), 1 (powinno być 1), -1 (powinno być 0)

                # jeśli error != 0, przesuwamy granicę decyzyjną w dobrym kierunku:
                self.weights += self.lr * error * xi  # aktualizacja wag
                self.bias += self.lr * error  # aktualizacja biasu

                if error != 0:  # jeśli źle sklasyfikowano tę próbkę
                    errors += 1  # zwiększ licznik błędów w epoce

            self.errors_per_epoch.append(errors)  # zapisz wynik epoki (do analizy zbieżności)

            if errors == 0:  # jeśli w całej epoce zero błędów -- model nauczył się danych
                print(f"Konwergencja po {epoch + 1} epokach!")
                break  # early stopping -- nie ma sensu trenować dalej

        return self  # zwróć wytrenowany model (umożliwia łańcuchowanie: model.fit(...).predict(...))


def verify_gate(name, X, y, model):
    """Pomocnicza funkcja: trenuje model i sprawdza, czy predykcje są poprawne."""
    print(f"=== BRAMKA {name} ===")  # nagłówek wyniku (AND albo OR)
    model.fit(X, y)  # ucz model na danych tej bramki
    print(f"Wagi: {model.weights}, Bias: {model.bias:.3f}")  # pokaż nauczone parametry
    print("Predykcje:")
    for xi, yi in zip(X, y):  # przejdź po każdej próbce testowej
        pred = model.predict(xi.reshape(1, -1))[0]  # reshape(1,-1) bo predict oczekuje macierzy 2D
        status = "OK" if pred == yi else "BŁĄD"  # porównaj predykcję z oczekiwaną wartością
        print(f"  {xi} --> {pred} (oczekiwane: {yi}) [{status}]")
    print()  # pusta linia dla czytelności


# --- BRAMKA AND ---
# macierz X: 4 kombinacje wejść (0/1, 0/1)
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
# wektor y: oczekiwane wyjście AND (1 tylko gdy oba wejścia = 1)
y_and = np.array([0, 0, 0, 1])

perceptron_and = Perceptron(n_features=2)  # tworzymy nowy, pusty model (2 wejścia)
verify_gate("AND", X_and, y_and, perceptron_and)  # uczymy i weryfikujemy


# --- BRAMKA OR ---
# te same wejścia co wyżej, ale inna logika wyjścia
X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
# OR daje 1, gdy przynajmniej jedno wejście = 1
y_or = np.array([0, 1, 1, 1])

perceptron_or = Perceptron(n_features=2)  # osobny model -- uczy się innej granicy decyzyjnej
verify_gate("OR", X_or, y_or, perceptron_or)

# Zadanie 2 -- Funkcje aktywacji
Zaimplementuj i zwizualizuj 4 funkcje aktywacji: sigmoid, tanh, ReLU, Leaky ReLU.

Wymagania:
- Zaimplementuj każdą funkcję i jej pochodną
- Narysuj 4 wykresy (2x2) z funkcją i pochodną na jednym
- Zakres z: od -5 do 5
- Oznacz zakres wartości na każdym wykresie

Oczekiwany wynik:
- 4 wykresy z funkcjami aktywacji i ich pochodnymi

In [ ]:
# Funkcje aktywacji -- implementacja, pochodne i wizualizacja
import numpy as np
import matplotlib.pyplot as plt


def sigmoid(z):
    """Sigmoid: σ(z) = 1 / (1 + e^(-z)). Zakres: (0, 1)."""
    return 1 / (1 + np.exp(-z))


def sigmoid_derivative(z):
    """Pochodna sigmoid: σ'(z) = σ(z) * (1 - σ(z))."""
    s = sigmoid(z)
    return s * (1 - s)


def tanh_fn(z):
    """Tanh: tanh(z). Zakres: (-1, 1)."""
    return np.tanh(z)


def tanh_derivative(z):
    """Pochodna tanh: 1 - tanh(z)^2."""
    return 1 - np.tanh(z) ** 2


def relu(z):
    """ReLU: max(0, z). Zakres: [0, +∞)."""
    return np.maximum(0, z)


def relu_derivative(z):
    """Pochodna ReLU: 1 dla z > 0, 0 w pozostałych przypadkach."""
    return np.where(z > 0, 1, 0)


def leaky_relu(z, alpha=0.01):
    """Leaky ReLU: z dla z > 0, alpha * z w przeciwnym razie."""
    return np.where(z > 0, z, alpha * z)


def leaky_relu_derivative(z, alpha=0.01):
    """Pochodna Leaky ReLU: 1 dla z > 0, alpha w przeciwnym razie."""
    return np.where(z > 0, 1, alpha)


# Zakres argumentu z -- wspólny dla wszystkich wykresów
z = np.linspace(-5, 5, 200)

# Układ 2x2 -- po jednym wykresie na każdą funkcję aktywacji
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Konfiguracja wykresów: (funkcja, pochodna, tytuł, zakres y, opis zakresu)
plots = [
    (sigmoid, sigmoid_derivative, "Sigmoid", (-0.1, 1.1), "Zakres: (0, 1)"),
    (tanh_fn, tanh_derivative, "Tanh", (-1.1, 1.1), "Zakres: (-1, 1)"),
    (relu, relu_derivative, "ReLU", (-0.5, 5.5), "Zakres: [0, +∞)"),
    (leaky_relu, leaky_relu_derivative, "Leaky ReLU (α=0.01)", (-0.5, 5.5), "Zakres: (-∞, +∞)"),
]

for ax, (fn, deriv, title, ylim, range_label) in zip(axes.flat, plots):
    # Niebieska linia ciągła -- funkcja aktywacji
    ax.plot(z, fn(z), "b-", linewidth=2, label="Funkcja")
    # Czerwona linia przerywana -- pochodna (potrzebna w backpropagation)
    ax.plot(z, deriv(z), "r--", linewidth=2, label="Pochodna")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("z")
    ax.set_ylabel("f(z)")
    ax.set_xlim(-5, 5)
    ax.set_ylim(*ylim)
    # Oznaczenie zakresu wartości na wykresie (wymaganie z zadania)
    ax.text(0.03, 0.95, range_label, transform=ax.transAxes, fontsize=10, va="top")
    ax.axhline(y=0, color="gray", linestyle=":", alpha=0.5)
    ax.axvline(x=0, color="gray", linestyle=":", alpha=0.5)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Funkcje aktywacji i ich pochodne", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Zadanie 3 -- Softmax ręcznie
Zaimplementuj funkcję softmax i przetestuj na 3 przykładach.

Wymagania:
- Implementacja softmax z numeryczną stabilnością (odejmij max)
- Test 1: z = [2.0, 1.0, 0.1]
- Test 2: z = [1.0, 1.0, 1.0] (równomierne)
- Test 3: z = [10.0, 0.0, -10.0] (dominująca klasa)
- Sprawdź, że suma prawdopodobieństw = 1.0
- Narysuj barplot dla każdego testu

Oczekiwany wynik:
- 3 barploty z rozkładami prawdopodobieństwa

In [ ]:
# Softmax od zera -- 3 testy z polecenia
import numpy as np
import matplotlib.pyplot as plt


def softmax(z):
    """Softmax: zamienia wektor logitów z na rozkład prawdopodobieństwa (suma = 1)."""
    z = np.asarray(z, dtype=float)  # upewniamy się, że pracujemy na tablicy NumPy
    # odejmujemy max(z) przed exp -- stabilność numeryczna (unikamy overflow)
    exp_z = np.exp(z - np.max(z))
    return exp_z / exp_z.sum()  # normalizacja: każda wartość / suma wszystkich


# 3 przykłady wyłącznie z treści zadania (PDF)
tests = [
    {
        "name": "Test 1",
        "desc": "z = [2.0, 1.0, 0.1]",
        "z": np.array([2.0, 1.0, 0.1]),
    },
    {
        "name": "Test 2",
        "desc": "z = [1.0, 1.0, 1.0] (równomierne)",
        "z": np.array([1.0, 1.0, 1.0]),
    },
    {
        "name": "Test 3",
        "desc": "z = [10.0, 0.0, -10.0] (dominująca klasa)",
        "z": np.array([10.0, 0.0, -10.0]),
    },
]

class_labels = ["Klasa 0", "Klasa 1", "Klasa 2"]
colors = ["#2196F3", "#4CAF50", "#FF9800"]

# 3 barploty obok siebie -- po jednym na każdy test
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, test in zip(axes, tests):
    z = test["z"]
    probs = softmax(z)  # oblicz rozkład prawdopodobieństwa dla danego wektora z

    # Weryfikacja numeryczna -- suma prawdopodobieństw musi wynosić 1.0
    print(f"=== {test['name']}: {test['desc']} ===")
    print(f"z: {z}")
    for label, p in zip(class_labels, probs):
        print(f"  {label}: {p:.6f}")
    sum_ok = np.isclose(probs.sum(), 1.0)
    print(f"Suma prawdopodobieństw: {probs.sum():.6f} [{'OK' if sum_ok else 'BŁĄD'}]")
    print()

    # Barplot rozkładu prawdopodobieństwa
    ax.bar(class_labels, probs, color=colors)
    ax.set_title(test["name"], fontsize=13)
    ax.set_ylabel("Prawdopodobieństwo")
    ax.set_ylim(0, 1.05)
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f"{p:.3f}", ha="center", fontsize=11)
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("Softmax -- rozkłady prawdopodobieństwa", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

# Zadanie 4 -- MLPClassifier na Iris
Wytrenuj MLPClassifier na datasecie Iris.

Wymagania:
- Architektura: (10, 5), activation='relu', solver='adam'
- Standaryzacja danych (StandardScaler)
- Oblicz accuracy, wyświetl classification_report
- Narysuj krzywą uczenia (loss curve)

Oczekiwany wynik:
- Raport klasyfikacji i wykres loss

In [ ]:
# MLPClassifier -- klasyfikacja Iris (sieć neuronowa sklearn)
import warnings
warnings.filterwarnings("ignore")  # wyłącza komunikaty RuntimeWarning / ConvergenceWarning sklearn

import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

# Wczytanie danych Iris (4 cechy, 3 klasy)
iris = load_iris()
X, y = iris.data, iris.target

print("Kształt danych:", X.shape)
print("Klasy:", iris.target_names)
print("Proporcje klas:", np.bincount(y))
print()

# Podział train/test 80/20 ze stratyfikacją (zachowujemy proporcje klas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standaryzacja -- MLP wymaga cech w podobnej skali
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # uczymy skaler tylko na train
X_test_sc = scaler.transform(X_test)  # ten sam skaler stosujemy na test

# MLP: 4 wejścia --> 10 neuronów --> 5 neuronów --> 3 klasy
mlp = MLPClassifier(
    hidden_layer_sizes=(10, 5),  # 2 warstwy ukryte
    activation="relu",  # ReLU w warstwach ukrytych
    solver="adam",  # optymalizator Adam
    max_iter=500,  # maksymalna liczba epok
    random_state=42,
    early_stopping=False,  # pełny trening + pełna krzywa loss
)

start = time.time()
mlp.fit(X_train_sc, y_train)
train_time = time.time() - start

y_pred = mlp.predict(X_test_sc)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Czas treningu: {train_time:.3f}s")
print(f"Liczba epok: {mlp.n_iter_}")
print(f"Architektura: {X_train.shape[1]} --> {mlp.hidden_layer_sizes} --> {len(iris.target_names)}")
print()
print("Raport klasyfikacji:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Krzywa uczenia (loss curve) -- spadek błędu cross-entropy w kolejnych epokach
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mlp.loss_curve_, linewidth=2, color="#2196F3")
ax.set_xlabel("Epoka", fontsize=12)
ax.set_ylabel("Loss (Cross-Entropy)", fontsize=12)
ax.set_title("Krzywa uczenia MLP na Iris", fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_yscale("log")  # skala log -- lepiej widać spadek loss na końcu
plt.tight_layout()
plt.show()

print(f"Początkowy loss: {mlp.loss_curve_[0]:.4f}")
print(f"Końcowy loss:    {mlp.loss_curve_[-1]:.4f}")

# Zadanie 5 -- Liczba parametrów
Napisz funkcję count_parameters() i oblicz liczbę parametrów dla 5 architektur.

Wymagania:
- Architektury: (4,3), (4,10,3), (784,128,10), (784,256,128,10), (784,512,256,128,64,10)
- Wyświetl detale: wagi + biasy per warstwa
- Narysuj barplot z całkowitą liczbą parametrów

Oczekiwany wynik:
- Tabela i wykres

In [ ]:
# Liczba parametrów w sieci neuronowej (wagi + biasy)
import matplotlib.pyplot as plt
import pandas as pd


def count_parameters(layer_sizes):
    """Oblicza liczbę parametrów (wagi + biasy) w MLP dla podanej architektury."""
    total = 0
    details = []

    # Każda para kolejnych warstw: warstwa i --> warstwa i+1
    for i in range(len(layer_sizes) - 1):
        n_weights = layer_sizes[i] * layer_sizes[i + 1]  # macierz wag
        n_biases = layer_sizes[i + 1]  # jeden bias na neuron w warstwie docelowej
        layer_total = n_weights + n_biases
        total += layer_total
        details.append({
            "warstwa": f"{layer_sizes[i]} --> {layer_sizes[i + 1]}",
            "wagi": n_weights,
            "biasy": n_biases,
            "razem": layer_total,
        })

    # Wypisanie szczegółów per warstwa
    print("Architektura:", " --> ".join(map(str, layer_sizes)))
    print("-" * 55)
    for d in details:
        print(
            f"  {d['warstwa']:>12s}: {d['wagi']:>8d} wag + "
            f"{d['biasy']:>4d} biasów = {d['razem']:>8d}"
        )
    print("-" * 55)
    print(f"  RAZEM: {total:>8d} parametrów\n")

    return total, details


# 5 architektur z polecenia (pełna ścieżka: wejście --> ukryte --> wyjście)
architectures = {
    "(4, 3)": [4, 3],
    "(4, 10, 3)": [4, 10, 3],
    "(784, 128, 10)": [784, 128, 10],
    "(784, 256, 128, 10)": [784, 256, 128, 10],
    "(784, 512, 256, 128, 64, 10)": [784, 512, 256, 128, 64, 10],
}

summary_rows = []
for name, layer_sizes in architectures.items():
    print(f"=== {name} ===")
    total, _ = count_parameters(layer_sizes)
    summary_rows.append({"Architektura": name, "Parametry": total})

# Tabela zbiorcza
summary_df = pd.DataFrame(summary_rows)
print("=== PODSUMOWANIE ===")
print(summary_df.to_string(index=False))

# Barplot — porównanie całkowitej liczby parametrów
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    summary_df["Architektura"],
    summary_df["Parametry"],
    color=["#4CAF50", "#2196F3", "#FF9800", "#F44336", "#9C27B0"],
)
ax.set_xlabel("Architektura", fontsize=12)
ax.set_ylabel("Liczba parametrów", fontsize=12)
ax.set_title("Liczba parametrów w zależności od architektury MLP", fontsize=14)
ax.tick_params(axis="x", rotation=15)
ax.grid(True, axis="y", alpha=0.3)

# Etykiety wartości na słupkach
for bar, value in zip(bars, summary_df["Parametry"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

# Zadanie 6 -- Porównanie aktywacji w MLP
Porównaj różne funkcje aktywacji w MLPClassifier na Breast Cancer.

Wymagania:
- Aktywacje: 'relu', 'tanh', 'logistic' (sigmoid)
- Architektura: (64, 32), solver='adam', max_iter=500
- Walidacja krzyżowa (cv=5)
- Narysuj bar chart z accuracy per aktywacja

Oczekiwany wynik:
- Porównanie 3 aktywacji

In [ ]:
# Porównanie aktywacji MLP -- Breast Cancer + walidacja krzyżowa
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

# Wczytanie danych Breast Cancer (30 cech, klasyfikacja binarna)
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

print("Kształt danych:", X.shape)
print("Klasy:", cancer.target_names)
print("Proporcje klas:", np.bincount(y))
print()

# 3 funkcje aktywacji do porownania (z polecenia)
activations = ["relu", "tanh", "logistic"]
cv_scores = {}  # srednia accuracy per aktywacja

for activation in activations:
    # Pipeline: skalowanie + MLP w jednym obiekcie (skaler uczony osobno w kazdym foldzie CV)
    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation=activation,
                solver="adam",
                max_iter=500,
                random_state=42,
                early_stopping=False,
            ),
        ),
    ])

    # Walidacja krzyzowa cv=5 -- 5 podzialow, srednia accuracy
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    cv_scores[activation] = scores

    print(f"{activation:>10s}: CV accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

# Bar chart -- porownanie sredniej accuracy per aktywacja
names = list(cv_scores.keys())
means = [cv_scores[a].mean() for a in names]
stds = [cv_scores[a].std() for a in names]
colors = ["#4CAF50", "#2196F3", "#FF9800"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, means, yerr=stds, capsize=6, color=colors, alpha=0.85)
ax.set_xlabel("Funkcja aktywacji", fontsize=12)
ax.set_ylabel("Accuracy (CV, cv=5)", fontsize=12)
ax.set_title("MLP na Breast Cancer -- porownanie aktywacji", fontsize=14)
ax.set_ylim(0.9, 1.0)
ax.grid(True, axis="y", alpha=0.3)

# Etykiety wartosci na slupkach
for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"{mean:.3f}",
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.tight_layout()
plt.show()

best_activation = max(cv_scores, key=lambda a: cv_scores[a].mean())
print(f"\nNajlepsza aktywacja: {best_activation} (CV accuracy = {cv_scores[best_activation].mean():.4f})")

# Zadanie 7 -- MLPRegressor na California Housing
Wytrenuj MLPRegressor na próbce 3000 obserwacji z California Housing.

Wymagania:
- Architektura: (32, 16), early_stopping=True
- Oblicz RMSE, MAE, R2
- Narysuj wykres predykcje vs rzeczywiste
- Narysuj loss curve

Oczekiwany wynik:
- Metryki regresji i 2 wykresy

In [ ]:
# MLPRegressor -- regresja na California Housing (probka 3000)
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Wczytanie California Housing (cechy demograficzne -> mediana ceny domu)
housing = fetch_california_housing()
X, y = housing.data, housing.target

# Probka 3000 obserwacji (wymaganie z zadania)
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=3000, replace=False)
X_sample, y_sample = X[sample_idx], y[sample_idx]

print("Probka:", X_sample.shape)
print()

# Podzial train/test 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42
)

# Standaryzacja cech
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# MLPRegressor -- tylko parametry z polecenia: architektura + early stopping
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    early_stopping=True,
)

mlp_reg.fit(X_train_sc, y_train)
y_pred = mlp_reg.predict(X_test_sc)

# Metryki regresji
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")
print(f"Epoki: {mlp_reg.n_iter_}")
print()

# 2 wykresy: predykcje vs rzeczywiste + krzywa loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Wykres 1 -- predykcje vs wartosci rzeczywiste
ax1.scatter(y_test, y_pred, alpha=0.5, s=20, color="#2196F3")
ax1.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2,
    label="Idealna predykcja",
)
ax1.set_xlabel("Wartosci rzeczywiste", fontsize=12)
ax1.set_ylabel("Predykcje MLP", fontsize=12)
ax1.set_title(f"MLP Regressor (R2={r2:.3f})", fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Wykres 2 -- krzywa uczenia (loss curve)
ax2.plot(mlp_reg.loss_curve_, linewidth=2, color="#4CAF50")
ax2.set_xlabel("Epoka", fontsize=12)
ax2.set_ylabel("Loss (MSE)", fontsize=12)
ax2.set_title("Krzywa uczenia -- MLPRegressor", fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Zadanie 8 -- Early stopping
Porównaj MLP z early_stopping=True vs False na Breast Cancer.

Wymagania:
- Architektura: (64, 32), max_iter=1000
- Zmierz czas treningu i liczbę epok dla obu wariantów
- Porównaj accuracy na zbiorze testowym
- Narysuj loss curve dla obu wariantów na jednym wykresie

Oczekiwany wynik:
- Porównanie czasu, epok i accuracy

In [ ]:
# Early stopping -- porownanie True vs False (Breast Cancer)
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Wczytanie Breast Cancer
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Podzial train/test 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standaryzacja
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Dwa warianty -- jedyna roznica: early_stopping
variants = {
    "early_stopping=True": True,
    "early_stopping=False": False,
}

results = {}

for label, use_early_stopping in variants.items():
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=1000,
        early_stopping=use_early_stopping,
    )

    start = time.time()
    mlp.fit(X_train_sc, y_train)
    train_time = time.time() - start

    y_pred = mlp.predict(X_test_sc)
    acc = accuracy_score(y_test, y_pred)

    results[label] = {
        "model": mlp,
        "epochs": mlp.n_iter_,
        "time": train_time,
        "accuracy": acc,
    }

    print(f"=== {label} ===")
    print(f"Czas treningu: {train_time:.3f}s")
    print(f"Liczba epok:   {mlp.n_iter_}")
    print(f"Accuracy:      {acc:.4f}")
    print()

# Loss curve obu wariantow na jednym wykresie
fig, ax = plt.subplots(figsize=(10, 6))

colors = {"early_stopping=True": "#2196F3", "early_stopping=False": "#FF9800"}

for label, r in results.items():
    ax.plot(r["model"].loss_curve_, linewidth=2, label=label, color=colors[label])

ax.set_xlabel("Epoka", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Early stopping -- porownanie krzywych uczenia (Breast Cancer)", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Podsumowanie
print("=== PODSUMOWANIE ===")
for label, r in results.items():
    print(
        f"{label:>22s}: epoki={r['epochs']:>4d}, "
        f"czas={r['time']:.3f}s, accuracy={r['accuracy']:.4f}"
    )

# Zadanie 11 -- MLP vs Random Forest na make_moons
Porównaj granice decyzyjne MLP i Random Forest na syntetycznych danych.

Wymagania:
- Wygeneruj make_moons(n_samples=500, noise=0.3)
- Wytrenuj MLP (20, 10) i Random Forest (100 drzew)
- Narysuj granice decyzyjne obu modeli obok siebie
- Porównaj accuracy, czas treningu

Oczekiwany wynik:
- 2 wykresy granic decyzyjnych i porównanie metryk

In [ ]:
# Zadanie 11 -- MLP vs Random Forest na make_moons (PDF sekcja 6.1)
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Dane syntetyczne -- parametry z polecenia PDF
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# MLP wymaga skalowania; RF trenujemy na surowych cechach
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# 2. Modele -- jedyne roznice miedzy nimi to architektura vs liczba drzew
mlp = MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=2000, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 3. Trening + pomiar czasu
start_mlp = time.time()
mlp.fit(X_train_sc, y_train)
time_mlp = time.time() - start_mlp

start_rf = time.time()
rf.fit(X_train, y_train)
time_rf = time.time() - start_rf

acc_mlp = accuracy_score(y_test, mlp.predict(X_test_sc))
acc_rf = accuracy_score(y_test, rf.predict(X_test))

print("=== POROWNANIE METRYK (zbior testowy) ===")
print(f"MLP (20, 10):       accuracy={acc_mlp:.4f}, czas={time_mlp:.3f}s")
print(f"RF (100 drzew):     accuracy={acc_rf:.4f}, czas={time_rf:.3f}s")
print()

# 4. Wizualizacja granic decyzyjnych -- styl jak PDF (contourf + scatter)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
h = 0.02  # gestosc siatki

# --- MLP (skalowane dane) ---
x_min, x_max = X_train_sc[:, 0].min() - 1, X_train_sc[:, 0].max() + 1
y_min, y_max = X_train_sc[:, 1].min() - 1, X_train_sc[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z_mlp = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[0].contourf(xx, yy, Z_mlp, alpha=0.3, cmap="RdYlBu")
axes[0].scatter(
    X_train_sc[:, 0], X_train_sc[:, 1],
    c=y_train, cmap="RdYlBu", edgecolors="black", s=30,
)
axes[0].set_title(f"MLP (20, 10) | Czas: {time_mlp:.3f}s", fontsize=12)
axes[0].grid(True, alpha=0.2)
axes[0].annotate(
    f"Accuracy: {acc_mlp:.3f}",
    xy=(0.02, 0.98), xycoords="axes fraction", va="top",
    fontsize=11, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

# --- Random Forest (surowe dane) ---
x_min_rf, x_max_rf = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min_rf, y_max_rf = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx_rf, yy_rf = np.meshgrid(
    np.arange(x_min_rf, x_max_rf, h), np.arange(y_min_rf, y_max_rf, h)
)

Z_rf = rf.predict(np.c_[xx_rf.ravel(), yy_rf.ravel()]).reshape(xx_rf.shape)

axes[1].contourf(xx_rf, yy_rf, Z_rf, alpha=0.3, cmap="RdYlBu")
axes[1].scatter(
    X_train[:, 0], X_train[:, 1],
    c=y_train, cmap="RdYlBu", edgecolors="black", s=30,
)
axes[1].set_title(f"Random Forest (100 drzew) | Czas: {time_rf:.3f}s", fontsize=12)
axes[1].grid(True, alpha=0.2)
axes[1].annotate(
    f"Accuracy: {acc_rf:.3f}",
    xy=(0.02, 0.98), xycoords="axes fraction", va="top",
    fontsize=11, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)

plt.suptitle(
    "Porownanie granic decyzyjnych: MLP vs Random Forest na make_moons",
    fontsize=15, y=1.02,
)
plt.tight_layout()
plt.show()



# Zadanie dodatkowe -- Wine Quality: MLP vs Random Forest

Porównaj **3 gotowe architektury MLP** z **Random Forest** na Wine Quality.  
**Nie wymyślasz parametrów** — wszystko jest w tabeli poniżej (jak w Zadaniach 6, 7, 11).

### Dataset (z lekcji 23)

| Element | Wartość |
|---------|---------|
| Plik | `winequality-red.csv` (UCI) |
| **Próbka** | **800 wierszy** (`random_state=42`) — celowo mniej niż pełne 1599, żeby CV było szybkie |
| Cechy | 11 kolumn chemicznych (bez `quality`) |
| Target | `quality >= 7` → 1 („good”), inaczej 0 |

*Pełny zbiór ma ~1600 wierszy — to nadal mały dataset tabularny (mniej niż próbka 3000 w Zadaniu 7). Tu bierzemy 800, żeby skupić się na porównaniu modeli, nie na czasie obliczeń.*

### Co testujesz (stała lista — 3 MLP + 1 RF)

| Model | `hidden_layer_sizes` | Po co w tym zadaniu |
|-------|----------------------|---------------------|
| MLP mały | `(32,)` | 1 warstwa — reguła kciuka z PDF: zacznij od 16–64 neuronów |
| MLP średni | `(32, 16)` | 2 warstwy — ta sama architektura co MLP w benchmarku lekcji (PDF, Zad. 15) |
| MLP duży | `(64, 32)` | 2 warstwy — jak Zadania 6 i 8 (Breast Cancer) |
| Random Forest | 100 drzew | Jak Zadanie 11 i PDF sekcja 5.1 — **baseline do porównania** |

### Parametry wspólne (identyczne dla wszystkich MLP)

| Parametr | Wartość | Skąd |
|----------|---------|------|
| `activation` | `'relu'` | Domyślny standard w lekcji |
| `solver` | `'adam'` | Zadania 4, 6, PDF |
| `max_iter` | `500` | Zadania 4, 6 |
| `early_stopping` | `True` | Zadania 7, 8, PDF Green IT |
| `random_state` | `42` | Cała lekcja |
| Skalowanie | `StandardScaler` w `Pipeline` | Zadania 4, 6 — MLP tego wymaga |
| Walidacja | `cross_val_score`, `cv=5` | Zadanie 6 |
| Podział testowy | 80/20, `stratify=y` | Zadanie 11 |

### Wymagania (co masz na wyjściu)

1. Wczytaj dane, binaryzuj target, weź **800 próbek**
2. Dla **3 architektur MLP** + **RF**: accuracy (CV=5), czas treningu CV
3. **Bar chart** — 4 słupki (3 MLP + RF), oś Y = accuracy
4. Wytrenuj na train **MLP `(32, 16)`** i **RF** — porównaj accuracy na **teście** (ten MLP jest „środkiem” z listy, nie musisz szukać „najlepszego”)

### Oczekiwany wynik

- Tabela 4 modeli (CV accuracy + czas)
- Wykres słupkowy CV
- Krótkie porównanie test accuracy: MLP `(32, 16)` vs RF



In [ ]:
# Wine Quality -- 3 architektury MLP vs Random Forest (parametry z tabeli w zadaniu)
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# --- STAŁE Z POLECENIA (nie zmieniaj -- to zamknięte zadanie) ---
SAMPLE_SIZE = 800          # probka -- mniej niz pelne 1599, szybsze CV
RANDOM_STATE = 42
CV_FOLDS = 5
RF_N_ESTIMATORS = 100      # jak Zadanie 11

# 3 architektury MLP -- kazda ma opis w markdownie zadania
MLP_ARCHITECTURES = {
    "MLP (32,)": (32,),
    "MLP (32, 16)": (32, 16),
    "MLP (64, 32)": (64, 32),
}

# Wspolne parametry MLP -- jedna konfiguracja dla wszystkich trzech
MLP_KWARGS = dict(
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=RANDOM_STATE,
    early_stopping=True,
)


def make_mlp_pipeline(hidden_layer_sizes):
    """Pipeline: skalowanie + MLP (wzorzec z Zadania 6)."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, **MLP_KWARGS)),
    ])


# 1. Wczytanie danych + binaryzacja (ten sam URL co lekcja 23)
url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)
wine = pd.read_csv(url, sep=";")
wine["target"] = (wine["quality"] >= 7).astype(int)

X = wine.drop(columns=["quality", "target"])
y = wine["target"]

# 2. Proba 800 wierszy -- wymaganie z zadania (Green IT / szybkie porownanie)
wine_sample = wine.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)
X = wine_sample.drop(columns=["quality", "target"])
y = wine_sample["target"]

print(f"Probka: {X.shape[0]} wierszy, {X.shape[1]} cech")
print(f"Proporcja klasy 'good' (quality >= 7): {y.mean():.3f}")
print()

# 3. Podzial train/test 80/20 (stratyfikacja -- klasa 'good' jest rzadka)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# 4. CV=5 dla kazdego modelu -- te same dane, ta sama metryka
cv_results = {}

print("=== WALIDACJA KRZYŻOWA (cv=5, accuracy) ===")
for name, hidden in MLP_ARCHITECTURES.items():
    model = make_mlp_pipeline(hidden)
    start = time.time()
    scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring="accuracy")
    elapsed = time.time() - start
    cv_results[name] = {"mean": scores.mean(), "std": scores.std(), "time": elapsed}
    print(f"{name:16s}  Acc={scores.mean():.4f}  std={scores.std():.4f}  czas={elapsed:.2f}s")

rf = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
start = time.time()
rf_scores = cross_val_score(rf, X_train, y_train, cv=CV_FOLDS, scoring="accuracy")
rf_elapsed = time.time() - start
cv_results["RF (100 drzew)"] = {"mean": rf_scores.mean(), "std": rf_scores.std(), "time": rf_elapsed}
print(f"{'RF (100 drzew)':16s}  Acc={rf_scores.mean():.4f}  std={rf_scores.std():.4f}  czas={rf_elapsed:.2f}s")
print()

# 5. Bar chart -- 4 modele obok siebie (jak Zadanie 6)
names = list(cv_results.keys())
means = [cv_results[n]["mean"] for n in names]
stds = [cv_results[n]["std"] for n in names]
colors = ["#9C27B0", "#7B1FA2", "#6A1B9A", "#2196F3"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, alpha=0.85)
ax.set_ylabel("Accuracy (CV, cv=5)", fontsize=12)
ax.set_title(f"Wine Quality ({SAMPLE_SIZE} probek) -- MLP vs Random Forest", fontsize=13)
ax.set_ylim(0.75, 1.0)
ax.grid(True, axis="y", alpha=0.3)

for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.008,
        f"{mean:.3f}",
        ha="center",
        fontsize=10,
    )

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

# 6. Test: MLP (32, 16) vs RF -- konkretne modele z polecenia (bez szukania "najlepszego")
mlp_fixed = make_mlp_pipeline((32, 16))

start = time.time()
mlp_fixed.fit(X_train, y_train)
mlp_time = time.time() - start
mlp_test_acc = accuracy_score(y_test, mlp_fixed.predict(X_test))

start = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - start
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))

print("=== ACCURACY NA ZBIORZE TESTOWYM ===")
print(f"{'Model':<20s}  {'Test accuracy':>14s}  {'Czas treningu':>14s}")
print("-" * 52)
print(f"{'MLP (32, 16)':<20s}  {mlp_test_acc:>14.4f}  {mlp_time:>13.3f}s")
print(f"{'RF (100 drzew)':<20s}  {rf_test_acc:>14.4f}  {rf_time:>13.3f}s")
print()
print("Wniosek: na danych tabelarycznych RF czesto >= MLP (tak pisze PDF lekcji 24).")

